# 3.10 Dizelerle Çalışma

Bu notebook, PDS Handbook (TR) web sayfasının **Türkçe Jupyter karşılığıdır** — aynı açıklamalar, ders notları ve kod örnekleri.

| | |
|---|---|
| **Web sayfası** | `chapters/03-pandas/10-working-with-strings.html` |
| **Çalıştırma** | JupyterLab, VS Code veya Colab — hücreleri **yukarıdan aşağı** sırayla (`Shift+Enter`) |
| **Bağımlılık** | Kod hücreleri birbirine bağlıdır; hata alırsanız önce üsttekileri çalıştırın |

> **Kaynak:** Jake VanderPlas, *Python Data Science Handbook* — Türkçe ders uyarlaması



Orijinal: Working With Strings

Python'un güçlü yanlarından biri dize verisini işleme ve dönüştürmedeki görece kolaylığıdır. Pandas bunun üzerine inşa edilir ve gerçek dünya verisiyle çalışırken (okuyun: temizlerken) gereken munging türünün önemli parçası olan kapsamlı bir vektörize dize işlemleri kümesi sunar. Bu bölümde Pandas dize işlemlerinden bazılarını ele alacağız; ardından internetten toplanmış çok dağınık bir tarif veri kümesini kısmen temizlemek için bunları kullanacağız.

## Pandas Dize İşlemlerine Giriş

Önceki bölümlerde NumPy ve Pandas gibi araçların aritmetik işlemleri genelleştirerek aynı işlemi birçok dizi elemanına kolayca ve hızlıca uygulayabildiğimizi gördük. Örneğin:


In [ ]:
# numpy_vektor.py
import numpy as np
x = np.array([2, 3, 5, 7, 11, 13])
x * 2



İşlemlerin bu vektörizasyonu, dizi verisi üzerinde çalışmanın sözdizimini sadeleştirir: artık dizinin boyutu veya şekliyle değil, yapılmasını istediğimiz işlemle ilgilenmemiz yeterlidir. Dize dizileri için NumPy bu kadar basit bir erişim sağlamaz; daha ayrıntılı bir döngü sözdizimi kullanmak zorunda kalırsınız:


In [ ]:
# liste_capitalize.py
data = ['peter', 'Paul', 'MARY', 'gUIDO']
[s.capitalize() for s in data]



Bazı verilerle bu yeterli olabilir; ancak eksik değer varsa kod kırılır — bu yüzden ek kontroller gerekir:


In [ ]:
# none_capitalize.py
data = ['peter', 'Paul', None, 'MARY', 'gUIDO']
[s if s is None else s.capitalize() for s in data]



Bu tür manuel yaklaşım yalnızca ayrıntılı ve kullanışsız değil, hata yapmaya da açıktır.

Pandas, vektörize dize işlemleri ihtiyacını ve dize içeren Series ile Index nesnelerinin str özniteliği aracılığıyla eksik veriyi doğru işleme ihtiyacını karşılar. Örneğin bu veriyle bir Pandas Series oluşturursak, eksik değer işleme yerleşik olan str.capitalize yöntemini doğrudan çağırabiliriz:


In [ ]:
# str_capitalize.py
import pandas as pd
names = pd.Series(data)
names.str.capitalize()



> **Not**
>

## Pandas Dize Yöntemleri Tabloları

Python'da dize manipülasyonunu iyi biliyorsanız Pandas dize sözdiziminin çoğu sezgiseldir; yöntemleri listelemek muhtemelen yeterlidir. Ayrıntılara girmeden önce buradan başlayacağız. Bu bölümdeki örnekler aşağıdaki Series nesnesini kullanır:


In [ ]:
# monte_series.py
monte = pd.Series(['Graham Chapman', 'John Cleese', 'Terry Gilliam',
                   'Eric Idle', 'Terry Jones', 'Michael Palin'])



### Python Dize Yöntemlerine Benzer Yöntemler

Python'un yerleşik dize yöntemlerinin neredeyse tamamı Pandas vektörize dize yöntemiyle eşlenir. Python dize yöntemlerini yansıtan Pandas str yöntemlerinin bir listesi:

Bunların dönüş tipleri farklıdır. lower gibi bazıları dize Series'i döndürür:


In [ ]:
# str_lower.py
monte.str.lower()



Bazıları sayı döndürür:


In [ ]:
# str_len.py
monte.str.len()



Veya Boolean değerler:


In [ ]:
# str_startswith.py
monte.str.startswith('T')



Diğerleri her eleman için liste veya başka bileşik değerler döndürür:


In [ ]:
# str_split.py
monte.str.split()



Tartışmaya devam ederken bu tür liste-dizisi nesneleri üzerinde daha fazla manipülasyon göreceğiz.

### Düzenli İfadeler Kullanan Yöntemler

Ayrıca her dize elemanının içeriğini incelemek için düzenli ifadeler (regex) kabul eden ve Python'un yerleşik re modülünün API kurallarının bir kısmını izleyen birkaç yöntem vardır:

Bunlarla geniş bir işlem yelpazesi yapılabilir. Örneğin her elemanın başındaki ardışık karakter grubunu isteyerek ilk adı çıkarabiliriz:


In [ ]:
# str_extract.py
monte.str.extract('([A-Za-z]+)', expand=False)



Veya daha karmaşık bir şey: ünsüzle başlayıp ünsüzle biten tüm adları bulmak için dize başı (^) ve dize sonu ($) regex karakterlerini kullanabiliriz:


In [ ]:
# str_findall.py
monte.str.findall(r'^[^AEIOU].*[^aeiou]$')



Düzenli ifadeleri Series veya DataFrame girişlerine özlü biçimde uygulayabilme, veri analizi ve temizliği için birçok olasılık açar.

### Çeşitli Yöntemler

Son olarak diğer kullanışlı işlemlere olanak tanıyan çeşitli yöntemler vardır:

#### Vektörize öğe erişimi ve dilimleme

Özellikle get ve slice işlemleri, her diziden vektörize öğe erişimine olanak tanır. Örneğin str.slice(0, 3) ile her dizinin ilk üç karakterini alabiliriz. Bu davranış Python'un normal indeksleme sözdizimiyle de kullanılabilir; df.str.slice(0, 3) ile df.str[0:3] eşdeğerdir:


In [ ]:
# str_slice.py
monte.str[0:3]



df.str.get(i) ve df.str[i] ile indeksleme de benzer şekilde çalışır.

Bu indeksleme yöntemleri, split ile döndürülen dizi elemanlarına da erişmenizi sağlar. Örneğin her girişin soyadını çıkarmak için split ile str indekslemesini birleştirebiliriz:


In [ ]:
# str_split_soyad.py
monte.str.split().str[-1]



### 🧪 Şimdi deneyin

🧪 Şimdi deneyin
      split + zincirleme str indekslemesi — basit ad-soyad ayırma:
      
        import pandas as pd
s = pd.Series([&quot;Ali Yılmaz&quot;, &quot;Ayşe Kaya&quot;, &quot;Mehmet Demir&quot;])
print(s.str.split().str[-1])

#### Gösterge değişkenleri

Biraz ek açıklama gerektiren bir diğer yöntem get_dummies'tır. Veriniz bir tür kodlanmış gösterge içeren bir sütuna sahipse kullanışlıdır. Örneğin A = "Amerika'da doğdu", B = "Birleşik Krallık'ta doğdu", C = "peynir sever", D = "spam sever" gibi kodlar içeren bir veri kümemiz olabilir:


In [ ]:
# full_monte.py
full_monte = pd.DataFrame({'name': monte,
                           'info': ['B|C|D', 'B|D', 'A|C',
                                    'B|D', 'B|C', 'B|C|D']})
full_monte



get_dummies rutini bu gösterge değişkenlerini bir DataFrame'e ayırmamıza izin verir:


In [ ]:
# get_dummies.py
full_monte['info'].str.get_dummies('|')



Bu işlemleri yapı taşları olarak kullanarak verinizi temizlerken sonsuz sayıda dize işleme prosedürü oluşturabilirsiniz.

Bu yöntemlere daha fazla girmeyeceğiz; Pandas çevrimiçi dokümantasyonundaki "Working with Text Data" bölümünü okumanızı veya 3.13 Kaynaklar bölümündeki kaynaklara başvurmanızı öneririm.

## Örnek: Tarif Veritabanı

Vektörize dize işlemleri dağınık gerçek dünya verisini temizlerken en faydalı hale gelir. Burada web'deki çeşitli kaynaklardan derlenmiş açık bir tarif veritabanı örneği üzerinden yürüyeceğiz. Hedefimiz tarif verisini malzeme listelerine ayrıştırmak; böylece eldeki malzemelere göre hızlıca tarif bulabiliriz. Derleme betikleri openrecipes deposunda; veritabanının en güncel bağlantısı da orada.

Veritabanı yaklaşık 30 MB'tır; aşağıdaki komutlarla indirilip açılabilir (notebook'taki yorum satırları):


In [ ]:
# repo = "https://raw.githubusercontent.com/jakevdp/open-recipe-data/master"
# !cd data && curl -O {repo}/recipeitems.json.gz
# !gunzip data/recipeitems.json.gz



Veritabanı JSON biçimindedir; pd.read_json ile okuruz (dosyanın her satırı bir JSON girişi olduğu için lines=True gerekir):


In [ ]:
# read_recipes.py
recipes = pd.read_json('data/recipeitems.json', lines=True)
recipes.shape



Yaklaşık 175.000 tarif ve 17 sütun görüyoruz. Ne olduğunu görmek için bir satıra bakalım:


In [ ]:
# recipes_iloc0.py
recipes.iloc[0]



Orada çok bilgi var; ancak çoğu web'den kazınmış veride tipik olduğu gibi çok dağınık biçimde. Özellikle malzeme listesi dize biçiminde; ilgilendiğimiz bilgiyi dikkatle çıkarmamız gerekecek. Malzemelere yakından bakarak başlayalım:


In [ ]:
# ingredients_len.py
recipes.ingredients.str.len().describe()



Malzeme listeleri ortalama 250 karakter; minimum 0, maksimum neredeyse 10.000 karakter!

Meraktan, en uzun malzeme listesine sahip tarif hangisi bakalım:


In [ ]:
# en_uzun_tarif.py
recipes.name[np.argmax(recipes.ingredients.str.len())]



Başka toplu keşifler de yapabiliriz; örneğin kaç tarifin kahvaltı yemeği olduğunu (küçük ve büyük harfi eşleştiren regex ile) görelim:


In [ ]:
# breakfast_count.py
recipes.description.str.contains('[Bb]reakfast').sum()



Veya kaç tarifte tarçın malzeme olarak geçiyor:


In [ ]:
# cinnamon_count.py
recipes.ingredients.str.contains('[Cc]innamon').sum()



Hatta "cinamon" yazım hatası yapan tarif var mı diye bakabiliriz:


In [ ]:
# cinamon_typo.py
recipes.ingredients.str.contains('[Cc]inamon').sum()



Pandas dize araçlarıyla mümkün olan veri keşfi türü budur. Python gerçekten bu tür data munging işlerinde parlar.

### 🧪 Şimdi deneyin

🧪 Şimdi deneyin
      Küçük bir metin Series'inde regex ile filtreleme deneyin:
      
        import pandas as pd
s = pd.Series([&quot;Python&quot;, &quot;pandas&quot;, &quot;NumPy&quot;, &quot;pydata&quot;])
print(s[s.str.contains(&quot;py&quot;, case=False, regex=True)])

### Basit Bir Tarif Önerici

Biraz daha ileri gidip basit bir tarif öneri sistemi yapalım: verilen malzeme listesi için hepsini kullanan tarifleri bulmak istiyoruz. Kavramsal olarak basit olsa da verinin heterojenliği görevi zorlaştırır; örneğin her satırdan temiz malzeme listesi çıkarmak için kolay bir işlem yok. Bu yüzden biraz hile yapacağız: yaygın malzemeler listesiyle başlayıp her tarifin malzeme listesinde geçip geçmediklerine bakacağız. Basitlik için şimdilik yalnızca ot ve baharatlarla sınırlı kalalım:


In [ ]:
# spice_list.py
spice_list = ['salt', 'pepper', 'oregano', 'sage', 'parsley',
              'rosemary', 'tarragon', 'thyme', 'paprika', 'cumin']



Ardından her malzemenin listede geçip geçmediğini gösteren True ve False değerlerinden oluşan Boolean bir DataFrame oluşturabiliriz:


In [ ]:
# spice_df.py
import re
spice_df = pd.DataFrame({
    spice: recipes.ingredients.str.contains(spice, re.IGNORECASE)
    for spice in spice_list})
spice_df.head()



Örnek olarak maydanoz, kırmızı biber ve tarhun kullanan bir tarif bulmak isteyelim. DataFrame'lerin query yöntemiyle (3.12 Performans ve Sorgu bölümünde ayrıntılı) çok hızlı hesaplayabiliriz:


In [ ]:
# spice_query.py
selection = spice_df.query('parsley & paprika & tarragon')
len(selection)



Bu kombinasyonla yalnızca 10 tarif buluyoruz. Seçimin döndürdüğü indeksi kullanarak bu tariflerin adlarını bulalım:


In [ ]:
# recipe_names.py
recipes.name[selection.index]



### 🧪 Şimdi deneyin

🧪 Şimdi deneyin
      Basit malzeme eşleştirmesi — elinizdeki malzemelerle tarif arama mantığını küçük ölçekte deneyin:
      
        import pandas as pd
recipes = pd.DataFrame({
    &quot;name&quot;: [&quot;Omlet&quot;, &quot;Çorba&quot;, &quot;Salata&quot;],
    &quot;ingredients&quot;: [&quot;yumurta, tuz, biber&quot;, &quot;tuz, soğan, biber&quot;, &quot;marul, limon&quot;]
})
have = [&quot;tuz&quot;, &quot;biber&quot;]
mask = recipes.ingredients.str.contains(&quot;|&quot;.join(have), case=False)
print(recipes[mask])

175.000 tariften 10'a indirdiğimize göre akşam yemeği için daha bilinçli karar verebiliriz.

### Tariflerle Daha İleriye

Umarım bu örnek Pandas dize yöntemlerinin etkin biçimde sağladığı veri temizleme işlemlerinin türü hakkında fikir vermiştir. Sağlam bir tarif öneri sistemi kurmak elbette çok daha fazla iş gerektirir! Her tariften tam malzeme listesi çıkarmak önemli bir parça olurdu; kullanılan formatların çeşitliliği bunu nispeten zaman alıcı kılar.

Veri biliminde gerçek dünya verisinin temizlenmesi ve munging'inin çoğu zaman işin büyük kısmını oluşturduğu gerçeğine işaret eder — Pandas bunu verimli yapmanıza yardımcı olacak araçları sunar.

> **Not**
>
